# LifeMate Agent — Master Colab & Google Drive Setup
**Dự án:** LifeMate Agent (Zalo AI Challenge 2026) — Team 3

Notebook này thiết lập môi trường chạy trên **Google Colab (GPU T4)** và cấu hình **tự động lưu Checkpoint & Submission lên Google Drive** vĩnh viễn, chống mất dữ liệu khi phiên làm việc bị ngắt kết nối (disconnect).

## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Kiểm tra GPU T4
!nvidia-smi

## 2. Thiết lập thư mục lưu trữ vĩnh viễn trên Drive (Persistent Storage)
> **Lưu ý hiệu năng:** Code và Data giải nén trên Colab SSD (`/content/LifeMateAgent`), còn Checkpoints, Adapters và Submissions được liên kết mềm (Symlink) sang Drive (`/content/drive/MyDrive/LifeMate_Artifacts/`). Cách này vừa đạt tốc độ đọc ghi tối đa, vừa bảo toàn 100% dữ liệu.

In [ ]:
import os
from pathlib import Path

# Thư mục lưu trữ trên Google Drive của bạn
DRIVE_DIR = Path("/content/drive/MyDrive/LifeMate_Artifacts")
DRIVE_CHECKPOINTS = DRIVE_DIR / "checkpoints"
DRIVE_SUBMISSIONS = DRIVE_DIR / "submissions"
DRIVE_REPORTS = DRIVE_DIR / "reports"

for p in [DRIVE_CHECKPOINTS, DRIVE_SUBMISSIONS, DRIVE_REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print(f"✅ Thư mục Google Drive đã sẵn sàng:\n  - Checkpoints: {DRIVE_CHECKPOINTS}\n  - Submissions: {DRIVE_SUBMISSIONS}")

## 3. Clone / Thiết lập Repo trên Colab SSD & Tạo Symlink sang Drive

In [ ]:
%cd /content

# Nếu bạn đã có repo trên Git hoặc tải file zip lên Drive:
# Cách A: Git clone (thay URL repo của đội bạn):
# !git clone https://github.com/your-username/LifeMateAgent.git
# %cd /content/LifeMateAgent

# Cách B: Hoặc tạo cấu trúc thư mục làm việc trực tiếp:
!mkdir -p /content/LifeMateAgent/outputs
%cd /content/LifeMateAgent

# TẠO SYMLINK: Mọi file lưu vào outputs/checkpoints sẽ tự động ghi thẳng vào Google Drive!
!rm -rf /content/LifeMateAgent/outputs/checkpoints
!rm -rf /content/LifeMateAgent/outputs/submissions

!ln -s /content/drive/MyDrive/LifeMate_Artifacts/checkpoints /content/LifeMateAgent/outputs/checkpoints
!ln -s /content/drive/MyDrive/LifeMate_Artifacts/submissions /content/LifeMateAgent/outputs/submissions

print("✅ Đã thiết lập Symlink: outputs/checkpoints và outputs/submissions -> Google Drive")

## 4. Cài đặt Dependencies cho Colab GPU

In [ ]:
# Cài đặt nhanh các thư viện cần thiết
!pip install -q faster-whisper transformers accelerate sentencepiece jiwer jsonlines soundfile librosa

## 5. Giải nén dữ liệu `train+public_test` từ Drive vào Colab SSD (Siêu tốc)

In [ ]:
# Mẹo: Nén train+public_test thành file zip trên Drive (ví dụ: data.zip)
# Sau đó giải nén chỉ mất vài giây vào Colab SSD:

DATA_ZIP_ON_DRIVE = "/content/drive/MyDrive/LifeMate_Artifacts/data.zip"

if os.path.exists(DATA_ZIP_ON_DRIVE):
    print("Đang giải nén dữ liệu từ Drive vào Colab SSD...")
    !unzip -q {DATA_ZIP_ON_DRIVE} -d /content/LifeMateAgent/
    print("✅ Dữ liệu đã sẵn sàng trên Colab SSD!")
else:
    print(f"ℹ️ Chưa thấy file {DATA_ZIP_ON_DRIVE}. Bạn có thể copy thư mục train+public_test vào /content/LifeMateAgent/")

## 6. Chạy thử nghiệm và Lưu Checkpoint

In [ ]:
# Thử nghiệm lưu 1 file checkpoint demo để kiểm tra xem Drive đã nhận chưa
import json, time

test_ckpt = {
    "timestamp": time.time(),
    "status": "colab_drive_sync_success",
    "model": "Qwen2.5-3B-Instruct",
    "asr": "faster-whisper-large-v3-turbo"
}

ckpt_path = "/content/LifeMateAgent/outputs/checkpoints/test_checkpoint.json"
with open(ckpt_path, "w", encoding="utf-8") as f:
    json.dump(test_ckpt, f, indent=2)

# Kiểm tra xem trên Google Drive đã có file chưa
drive_check = os.path.exists("/content/drive/MyDrive/LifeMate_Artifacts/checkpoints/test_checkpoint.json")
print(f"✅ Kiểm tra file trên Google Drive thật: {'THÀNH CÔNG (Đã thấy trên Drive)' if drive_check else 'CHƯA THẤY'}")

## 7. Các lệnh chạy nhanh Pipeline

In [ ]:
# Chạy EDA
!python scripts/run_eda.py

# Chạy Gold Replay Baseline trên Dev Split
!python scripts/run_baseline.py

# Chạy Chẩn đoán Phân rã Điểm mất (Loss Attribution)
!python -m src.part_a.loss_attribution --submission outputs/submissions/baseline_gold_replay.jsonl --split dev

# Kiểm tra tính toàn vẹn Submission
!python -m src.part_a.submission_linter --submission outputs/submissions/baseline_gold_replay.jsonl --split public_test